# Assignment 8: Linear Models and LASSO

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

## Part A. Regression: predicting NBA salaries

Salary is in millions of dollars. We use only the numeric columns as predictors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, lasso_path
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("https://bryantstats.github.io/math460/data/nba_salary2.csv").dropna()
y = df["Salary"]
X = df.drop(columns=["Salary"]).select_dtypes("number")
print(X.shape)
X.head()

### Question 1. Split the data

Split the data into 70% training and 30% testing. Use `random_state=1` so everyone gets the same split.

In [ ]:
# TODO: split X and y into X_train, X_test, y_train, y_test
print(X_train.shape, X_test.shape)

### Question 2. Ordinary linear regression

Train a linear regression on the training data. Report the **test RMSE** and **test $R^2$**. The helper function below computes both.

In [ ]:
def report(name, model):
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    print(f"{name:<20} test RMSE = {rmse:.3f}   test R2 = {r2_score(y_test, pred):.3f}")

# TODO: train a LinearRegression called `ols` on the training data
report("Linear regression", ols)

### Question 3. LASSO with cross-validation

LASSO adds the penalty $\lambda \sum_j |\beta_j|$ to the RSS (scikit-learn calls $\lambda$ `alpha`).

a. Build a pipeline that **standardizes** the predictors and then fits `LassoCV(cv=5, random_state=1, max_iter=50000)`. Train it and report its test RMSE and $R^2$.

b. What value of `alpha` did cross-validation choose?

c. How many of the predictors have a coefficient of exactly 0? Which predictors did LASSO keep?

In [ ]:
# TODO: build and train the pipeline; call it `lasso`
report("LASSO", lasso)

lasso_model = lasso[-1]
print("Chosen alpha:", round(lasso_model.alpha_, 4))
coefs = pd.Series(lasso_model.coef_, index=X.columns)
print("Coefficients equal to 0:", (coefs == 0).sum(), "of", len(coefs))
coefs[coefs != 0].sort_values(key=abs, ascending=False)

**Your answer:** *Write your answer here.*

### Question 4. The LASSO path

The plot shows how each coefficient changes as `alpha` grows. Run the cell, then answer:

a. What happens to the coefficients as `alpha` increases?

b. Name one predictor that stays in the model the longest (is the last to reach 0). Why might it be so important for salary?

In [ ]:
X_std = StandardScaler().fit_transform(X_train)
alphas, path, _ = lasso_path(X_std, y_train, max_iter=50000)

plt.figure(figsize=(9, 5))
for j in range(path.shape[0]):
    plt.plot(np.log10(alphas), path[j], lw=1)
plt.axvline(np.log10(lasso_model.alpha_), color="black", ls="--", label="alpha chosen by CV")
plt.xlabel("log10(alpha)")
plt.ylabel("coefficient (standardized predictors)")
plt.title("LASSO path")
plt.legend()
plt.show()

last = pd.Series((np.abs(path) > 0).sum(axis=1), index=X.columns).sort_values(ascending=False)
print("Number of alphas (out of 100) at which each predictor is nonzero:")
print(last.head(5))

**Your answer:** *Write your answer here.*

### Question 5. Ridge regression

Ridge uses the penalty $\lambda \sum_j \beta_j^2$ instead. Train a standardized `RidgeCV(alphas=np.logspace(-3, 3, 50))` and report its test performance. Does Ridge set any coefficients to exactly 0?

In [ ]:
# TODO: build and train the ridge pipeline; call it `ridge`
report("Ridge", ridge)
print("Ridge coefficients equal to 0:", (ridge[-1].coef_ == 0).sum())

**Your answer:** *Write your answer here.*

## Part B. Classification: breast cancer diagnosis

The target is `diagnosis` (M = malignant = 1, B = benign = 0).

In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score

bc = pd.read_csv("https://bryantstats.github.io/math460/data/breast_cancer.csv")
bc = bc.drop(columns=["id", "Unnamed: 32"], errors="ignore")
yb = (bc["diagnosis"] == "M").astype(int)
Xb = bc.drop(columns=["diagnosis"])
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.3, random_state=1, stratify=yb)
print(Xb.shape)

### Question 6. Logistic regression vs. LASSO logistic regression

a. Train a standardized logistic regression **without a penalty** (`LogisticRegression(penalty=None, max_iter=10000)`).

b. Train a standardized **L1-penalized** logistic regression with cross-validation: `LogisticRegressionCV(penalty="l1", solver="liblinear", Cs=20, cv=5)`.

c. Compare the two on test accuracy and on the number of predictors each model uses. Which model would you choose, and why?

In [ ]:
# TODO: train `logit` (no penalty) and `logit_l1` (L1, cross-validated)

for name, m in [("No penalty", logit), ("L1 penalty", logit_l1)]:
    acc = accuracy_score(yb_test, m.predict(Xb_test))
    used = (m[-1].coef_ != 0).sum()
    print(f"{name:<12} test accuracy = {acc:.3f}   predictors used = {used} of {Xb.shape[1]}")

**Your answer:** *Write your answer here.*

### Question 7. Reflection

In two or three sentences: why must the predictors be **standardized** before fitting LASSO or Ridge, but not before fitting ordinary linear regression?

**Your answer:** *Write your answer here.*